# Background information on dataset

The dataset contains credit card payment data from October 2005, from a bank (a cash and credit card issuer) in Taiwan.

Among the total 25,000 observations, 5,529 observations (22.12%) are from cardholders with default payment.

The dataset contains a binary outcome variable — default payment (Yes = 1, No = 0) — and the following 23 explanatory variables:

- **X1** — Credit limit (NT dollar): includes both the individual consumer credit and their family (supplementary) credit
- **X2** — Gender (1 = male; 2 = female)
- **X3** — Education (1 = graduate school; 2 = university; 3 = high school; 4 = others)
- **X4** — Marital status (1 = married; 2 = single; 3 = others)
- **X5** — Age (years)

**X6–X11 — history of past payment (repayment status)**

Monthly repayment records tracked from April to September 2005:

| Variable | Month |
|---|---|
| X6 | September 2005 |
| X7 | August 2005 |
| X8 | July 2005 |
| X9 | June 2005 |
| X10 | May 2005 |
| X11 | April 2005 |

Measurement scale for repayment status:

| Value | Meaning |
|---|---|
| -1 | Pay duly (on time) |
| 1 | Payment delay for one month |
| 2 | Payment delay for two months |
| ... | ... |
| 8 | Payment delay for eight months |
| 9 | Payment delay for nine months and above |

**X12–X17 — amount of bill statement (NT dollar)**

| Variable | Month |
|---|---|
| X12 | September 2005 |
| X13 | August 2005 |
| X14 | July 2005 |
| X15 | June 2005 |
| X16 | May 2005 |
| X17 | April 2005 |

**X18–X23 — amount of previous payment (NT dollar)**

| Variable | Month |
|---|---|
| X18 | September 2005 |
| X19 | August 2005 |
| X20 | July 2005 |
| X21 | June 2005 |
| X22 | May 2005 |
| X23 | April 2005 |

In [1]:
import pandas as pd
from config import (
    CATEGORICAL_THRESHOLD,
    COL_NAME_MAP,
    DATA,
    EDUCATION_MAP,
    MARRIAGE_MAP,
)

# Read in data, clean up names and dtypes

In [2]:
df = pd.read_excel(DATA / "raw/default of credit card clients.xls")
df.columns = [
    f"{col_name}_{row_1}"
    for (col_name, row_1) in zip(df.columns, df.iloc[0, :], strict=True)
]
df = df.rename(columns=COL_NAME_MAP)
df = df.drop(0, axis=0).reset_index(drop=True)
df = df.convert_dtypes()
df.head(10)

,ID,X01_LIMIT_BAL,X02_SEX,X03_EDUCATION,X04_MARRIAGE,X05_AGE,X06_PAY_0,X07_PAY_2,X08_PAY_3,X09_PAY_4,X10_PAY_5,X11_PAY_6,X12_BILL_AMT1,X13_BILL_AMT2,X14_BILL_AMT3,X15_BILL_AMT4,X16_BILL_AMT5,X17_BILL_AMT6,X18_PAY_AMT1,X19_PAY_AMT2,X20_PAY_AMT3,X21_PAY_AMT4,X22_PAY_AMT5,X23_PAY_AMT6,Y_DEF_FLAG_1M
0,1,20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,0,0,46990,48233,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679,0
5,6,50000,1,1,2,37,0,0,0,0,0,0,64400,57069,57608,19394,19619,20024,2500,1815,657,1000,1000,800,0
6,7,500000,1,1,2,29,0,0,0,0,0,0,367965,412023,445007,542653,483003,473944,55000,40000,38000,20239,13750,13770,0
7,8,100000,2,2,2,23,0,-1,-1,0,0,-1,11876,380,601,221,-159,567,380,601,0,581,1687,1542,0
8,9,140000,2,3,1,28,0,0,2,0,0,0,11285,14096,12108,12211,11793,3719,3329,0,432,1000,1000,1000,0
9,10,20000,1,3,2,35,-2,-2,-2,-2,-1,-1,0,0,0,0,13007,13912,0,0,0,13007,1122,0,0


In [3]:
df.columns

Index(['ID', 'X01_LIMIT_BAL', 'X02_SEX', 'X03_EDUCATION', 'X04_MARRIAGE',
       'X05_AGE', 'X06_PAY_0', 'X07_PAY_2', 'X08_PAY_3', 'X09_PAY_4',
       'X10_PAY_5', 'X11_PAY_6', 'X12_BILL_AMT1', 'X13_BILL_AMT2',
       'X14_BILL_AMT3', 'X15_BILL_AMT4', 'X16_BILL_AMT5', 'X17_BILL_AMT6',
       'X18_PAY_AMT1', 'X19_PAY_AMT2', 'X20_PAY_AMT3', 'X21_PAY_AMT4',
       'X22_PAY_AMT5', 'X23_PAY_AMT6', 'Y_DEF_FLAG_1M'],
      dtype='str')

In [4]:
dtypes = df.dtypes
dtypes

ID               Int64
X01_LIMIT_BAL    Int64
X02_SEX          Int64
X03_EDUCATION    Int64
X04_MARRIAGE     Int64
X05_AGE          Int64
X06_PAY_0        Int64
X07_PAY_2        Int64
X08_PAY_3        Int64
X09_PAY_4        Int64
X10_PAY_5        Int64
X11_PAY_6        Int64
X12_BILL_AMT1    Int64
X13_BILL_AMT2    Int64
X14_BILL_AMT3    Int64
X15_BILL_AMT4    Int64
X16_BILL_AMT5    Int64
X17_BILL_AMT6    Int64
X18_PAY_AMT1     Int64
X19_PAY_AMT2     Int64
X20_PAY_AMT3     Int64
X21_PAY_AMT4     Int64
X22_PAY_AMT5     Int64
X23_PAY_AMT6     Int64
Y_DEF_FLAG_1M    Int64
dtype: object

In [5]:
info = df.info()
info

<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   ID             30000 non-null  Int64
 1   X01_LIMIT_BAL  30000 non-null  Int64
 2   X02_SEX        30000 non-null  Int64
 3   X03_EDUCATION  30000 non-null  Int64
 4   X04_MARRIAGE   30000 non-null  Int64
 5   X05_AGE        30000 non-null  Int64
 6   X06_PAY_0      30000 non-null  Int64
 7   X07_PAY_2      30000 non-null  Int64
 8   X08_PAY_3      30000 non-null  Int64
 9   X09_PAY_4      30000 non-null  Int64
 10  X10_PAY_5      30000 non-null  Int64
 11  X11_PAY_6      30000 non-null  Int64
 12  X12_BILL_AMT1  30000 non-null  Int64
 13  X13_BILL_AMT2  30000 non-null  Int64
 14  X14_BILL_AMT3  30000 non-null  Int64
 15  X15_BILL_AMT4  30000 non-null  Int64
 16  X16_BILL_AMT5  30000 non-null  Int64
 17  X17_BILL_AMT6  30000 non-null  Int64
 18  X18_PAY_AMT1   30000 non-null  Int64
 19  X19_PAY_AMT2   

In [6]:
print(f"Dataframe size: {df.shape}")

Dataframe size: (30000, 25)


In [7]:
profile = pd.DataFrame(
    {
        "dtype": df.dtypes,
        "%_missing": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(),
        "example": df.iloc[0],
    }
)
profile

,dtype,%_missing,n_unique,example
ID,Int64,0.0,30000,1
X01_LIMIT_BAL,Int64,0.0,81,20000
X02_SEX,Int64,0.0,2,2
X03_EDUCATION,Int64,0.0,7,2
X04_MARRIAGE,Int64,0.0,4,1
X05_AGE,Int64,0.0,56,24
X06_PAY_0,Int64,0.0,11,2
X07_PAY_2,Int64,0.0,11,2
X08_PAY_3,Int64,0.0,11,-1
X09_PAY_4,Int64,0.0,11,-1


# Identify Categorical / Numerical variables

All variables are integer, so no need to worry about string variables. Just need to identify categorical / numerical variables

In [8]:
uniques = df.nunique().sort_values()
uniques

Y_DEF_FLAG_1M        2
X02_SEX              2
X04_MARRIAGE         4
X03_EDUCATION        7
X10_PAY_5           10
X11_PAY_6           10
X06_PAY_0           11
X07_PAY_2           11
X08_PAY_3           11
X09_PAY_4           11
X05_AGE             56
X01_LIMIT_BAL       81
X22_PAY_AMT5      6897
X21_PAY_AMT4      6937
X23_PAY_AMT6      6939
X20_PAY_AMT3      7518
X19_PAY_AMT2      7899
X18_PAY_AMT1      7943
X17_BILL_AMT6    20604
X16_BILL_AMT5    21010
X15_BILL_AMT4    21548
X14_BILL_AMT3    22026
X13_BILL_AMT2    22346
X12_BILL_AMT1    22723
ID               30000
dtype: int64

In [9]:
cat_vars = list(uniques[uniques <= CATEGORICAL_THRESHOLD].index.sort_values())
num_vars = list(uniques[uniques > CATEGORICAL_THRESHOLD].index.sort_values())
print(f"Low order variables: {cat_vars}\n")
print(f"Numerical variables: {num_vars}\n")

for c in cat_vars:
    vc = df[c].value_counts(dropna=False).sort_index().to_frame("n")
    vc["pct"] = (vc.n / len(df) * 100).round(1)
    print(f"\n--- {c} ---")
    print(vc)

Low order variables: ['X02_SEX', 'X03_EDUCATION', 'X04_MARRIAGE', 'X06_PAY_0', 'X07_PAY_2', 'X08_PAY_3', 'X09_PAY_4', 'X10_PAY_5', 'X11_PAY_6', 'Y_DEF_FLAG_1M']

Numerical variables: ['ID', 'X01_LIMIT_BAL', 'X05_AGE', 'X12_BILL_AMT1', 'X13_BILL_AMT2', 'X14_BILL_AMT3', 'X15_BILL_AMT4', 'X16_BILL_AMT5', 'X17_BILL_AMT6', 'X18_PAY_AMT1', 'X19_PAY_AMT2', 'X20_PAY_AMT3', 'X21_PAY_AMT4', 'X22_PAY_AMT5', 'X23_PAY_AMT6']


--- X02_SEX ---
             n   pct
X02_SEX             
1        11888  39.6
2        18112  60.4

--- X03_EDUCATION ---
                   n   pct
X03_EDUCATION             
0                 14   0.0
1              10585  35.3
2              14030  46.8
3               4917  16.4
4                123   0.4
5                280   0.9
6                 51   0.2

--- X04_MARRIAGE ---
                  n   pct
X04_MARRIAGE             
0                54   0.2
1             13659  45.5
2             15964  53.2
3               323   1.1

--- X06_PAY_0 ---
               n   

Conclusion: From defintions in data background section above, all low order variables should be treated as categorical. Variables X06_PAY_0 -> X11_PAY_6 variables are a combination of categorical (-1 for paid duly) and numerical (1 -> 9 months down). For now, investigate further as both categorical and numerical 

In [10]:
num_vars += [
    "X06_PAY_0",
    "X07_PAY_2",
    "X08_PAY_3",
    "X09_PAY_4",
    "X10_PAY_5",
    "X11_PAY_6",
]

# Transform catergorical variables

X04_MARRIAGE has 54 occurences of '0' which is unlisted in data dict. Overwrite these 
with '3' for 'Other'.  

X03_Education includes '0', '5' and'6' values which similarly do not appear in the data 
dictionary. Overwrite with '4' for 'Other'.  

PAY_*: the source information documents only −1 and 1–9 for these variables. Values −2 and 0 appear in the data with significant volumes across variables. From googling, the widely-used interpretation is that −2 = no consumption and 0 = revolving credit. Treat as numeric for the time being; the linear relationship between 1-9 months down will be treated as continuing for values -2, -1 and 0. This variable should be broken out into multiple variables or transformed in later versions of this model. Limitation of this treatment is that it assumes an ordering for codes whose meaning is unverified. 

In [11]:
df["X03_EDUCATION"] = df["X03_EDUCATION"].replace(EDUCATION_MAP)
df["X04_MARRIAGE"] = df["X04_MARRIAGE"].replace(MARRIAGE_MAP)
cat_vars = [
    cv
    for cv in cat_vars
    if cv
    not in [
        "X06_PAY_0",
        "X07_PAY_2",
        "X08_PAY_3",
        "X09_PAY_4",
        "X10_PAY_5",
        "X11_PAY_6",
    ]
]

for c in cat_vars:
    vc = df[c].value_counts(dropna=False).sort_index().to_frame("n")
    vc["pct"] = (vc.n / len(df) * 100).round(1)
    print(f"\n--- {c} ---")
    print(vc)


--- X02_SEX ---
             n   pct
X02_SEX             
1        11888  39.6
2        18112  60.4

--- X03_EDUCATION ---
                   n   pct
X03_EDUCATION             
1              10585  35.3
2              14030  46.8
3               4917  16.4
4                468   1.6

--- X04_MARRIAGE ---
                  n   pct
X04_MARRIAGE             
1             13659  45.5
2             15964  53.2
3               377   1.3

--- Y_DEF_FLAG_1M ---
                   n   pct
Y_DEF_FLAG_1M             
0              23364  77.9
1               6636  22.1


# Transform numerical variables

In [12]:
description_df = df[num_vars].describe()
description_df = pd.pivot_table(description_df.reset_index(), columns="index")
description_df

index,25%,50%,75%,count,max,mean,min,std
ID,7500.75,15000.5,22500.25,30000.0,30000.0,15000.5,1.0,8660.398374
X01_LIMIT_BAL,50000.0,140000.0,240000.0,30000.0,1000000.0,167484.322667,10000.0,129747.661567
X05_AGE,28.0,34.0,41.0,30000.0,79.0,35.4855,21.0,9.217904
X06_PAY_0,-1.0,0.0,0.0,30000.0,8.0,-0.0167,-2.0,1.123802
X07_PAY_2,-1.0,0.0,0.0,30000.0,8.0,-0.133767,-2.0,1.197186
X08_PAY_3,-1.0,0.0,0.0,30000.0,8.0,-0.1662,-2.0,1.196868
X09_PAY_4,-1.0,0.0,0.0,30000.0,8.0,-0.220667,-2.0,1.169139
X10_PAY_5,-1.0,0.0,0.0,30000.0,8.0,-0.2662,-2.0,1.133187
X11_PAY_6,-1.0,0.0,0.0,30000.0,8.0,-0.2911,-2.0,1.149988
X12_BILL_AMT1,3558.75,22381.5,67091.0,30000.0,964511.0,51223.3309,-165580.0,73635.860576


In [13]:
df.to_csv("../data/interim/taiwan_default_data_v0.1.csv", index=False)